In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
import numpy as np
from PIL import Image

# Dataset directory
data_dir = '/Users/kaivalyapuranik/Downloads/archive'

# Parameters
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 30
TEST_SIZE = 0.15  # 15% for testing

# Load image paths and labels
def load_image_paths_and_labels(base_dir):
    class_names = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    all_filepaths = []
    all_labels = []
    
    for idx, class_name in enumerate(class_names):
        class_folder = os.path.join(base_dir, class_name)
        file_names = [f for f in os.listdir(class_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        for file_name in file_names:
            all_filepaths.append(os.path.join(class_folder, file_name))
            all_labels.append(idx)  # numeric label
            
    return np.array(all_filepaths), np.array(all_labels), class_names

filepaths, labels, class_names = load_image_paths_and_labels(data_dir)
print(f"Total images: {len(filepaths)} | Classes: {class_names}")

# Split data
train_paths, test_paths, train_labels, test_labels = train_test_split(
    filepaths, labels, test_size=TEST_SIZE, stratify=labels, random_state=42
)

print(f"Train samples: {len(train_paths)} | Test samples: {len(test_paths)}")

# Preprocess function
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)  # handle jpeg/png
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0  # normalize to [0,1]
    return image, label

# TF datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# Model setup
base_model = tf.keras.applications.ResNet50(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze

# Number of classes
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

# Final activation and loss
if num_classes == 2:
    final_activation = 'sigmoid'
    final_units = 1
    loss_function = 'binary_crossentropy'
else:
    final_activation = 'softmax'
    final_units = num_classes
    loss_function = 'sparse_categorical_crossentropy'  # because labels are integers

# Build the model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(final_units, activation=final_activation)
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_function,
    metrics=['accuracy']
)

# Train
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds
)

# Save the model
model.save('glaucoma_detection_resnet50.h5')

# Evaluate & Predict
# Get test data arrays for predictions
test_images = []
test_labels_list = []

for image, label in test_ds.unbatch():
    test_images.append(image.numpy())
    test_labels_list.append(label.numpy())

test_images = np.array(test_images)
test_labels_list = np.array(test_labels_list)

# Predictions
predictions = model.predict(test_images)

if num_classes == 2:
    predicted_classes = (predictions > 0.5).astype("int32").flatten()
else:
    predicted_classes = np.argmax(predictions, axis=1)

# Classification report
report = classification_report(test_labels_list, predicted_classes, target_names=class_names)
print("Classification Report:\n", report)

# Final validation accuracy
val_acc = history.history['val_accuracy'][-1]
print(f"Final validation accuracy: {val_acc:.4f}")

Total images: 2400 | Classes: ['Benign', 'Malignant']
Train samples: 2040 | Test samples: 360
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 16s 0us/step
Number of classes: 2
Epoch 1/30


/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


64/64 ━━━━━━━━━━━━━━━━━━━━ 59s 868ms/step - accuracy: 0.5549 - loss: 0.6973 - val_accuracy: 0.7167 - val_loss: 0.6436
Epoch 2/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 845ms/step - accuracy: 0.6270 - loss: 0.6483 - val_accuracy: 0.6611 - val_loss: 0.6224
Epoch 3/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 837ms/step - accuracy: 0.6716 - loss: 0.6149 - val_accuracy: 0.7806 - val_loss: 0.5692
Epoch 4/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 53s 833ms/step - accuracy: 0.7113 - loss: 0.5775 - val_accuracy: 0.8139 - val_loss: 0.5325
Epoch 5/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 841ms/step - accuracy: 0.7078 - loss: 0.5644 - val_accuracy: 0.7722 - val_loss: 0.5198
Epoch 6/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 841ms/step - accuracy: 0.7490 - loss: 0.5252 - val_accuracy: 0.8222 - val_loss: 0.4763
Epoch 7/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 835ms/step - accuracy: 0.7696 - loss: 0.5041 - val_accuracy: 0.8028 - val_loss: 0.4675
Epoch 8/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 54s 842ms/step - accuracy: 0.7740 - loss: 0.4871 - val_accuracy: 0.833

12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 742ms/step
Classification Report:
               precision    recall  f1-score   support

      Benign       0.87      0.94      0.90       180
   Malignant       0.93      0.86      0.90       180

    accuracy                           0.90       360
   macro avg       0.90      0.90      0.90       360
weighted avg       0.90      0.90      0.90       360

Final validation accuracy: 0.9000
